# Project 16 — BROKEN notebook (change-point pitfalls)

Seeded bugs: a wrong-direction switch and treating a (potentially multimodal) tau posterior by its mean. Run it, read the diagnostics, fix each. Clean reference: `notebook.ipynb`; answer key: `BROKEN_BUGS.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate(); y = data['y']; t = data['truth']; T = data['T']
idx = np.arange(T)

### BUG 1 — wrong switch direction (rates swapped before/after tau).

`switch(tau < idx, ...)` (or swapping lam0/lam1) assigns the pre-shift rate to the post-shift region. The model still runs but fits a mirror-image change.

In [ ]:
with pm.Model() as model:
    tau = pm.DiscreteUniform('tau', lower=1, upper=T-1)
    lam0 = pm.Exponential('lam0', 1/5.)
    lam1 = pm.Exponential('lam1', 1/5.)
    # BUG 1: should be switch(tau > idx, lam0, lam1)
    rate = pm.math.switch(tau < idx, lam0, lam1)
    pm.Poisson('y', mu=rate, observed=y)
    # BUG 2: too few tune steps for the discrete+continuous geometry
    idata = pm.sample(draws=800, tune=100, chains=2, random_seed=RNG,
                      progressbar=False)

In [ ]:
print(az.summary(idata, var_names=['tau','lam0','lam1']))
print('true tau,lam0,lam1 =', t['tau'], t['lam0'], t['lam1'])
# Symptom: lam0/lam1 come out swapped relative to truth.

### BUG 3 — summarizing tau by its mean.

If the tau posterior is multimodal (or just skewed), the **mean** falls between peaks and points at a time the data do not support.

In [ ]:
tau_draws = idata.posterior['tau'].values.ravel()
# BUG 3: reporting the mean of a discrete (possibly multimodal) tau
print('reported tau =', tau_draws.mean())
fig, ax = plt.subplots(figsize=(7,3))
ax.hist(tau_draws, bins=np.arange(0, T+1)-0.5, color='#C44E52')
ax.axvline(tau_draws.mean(), color='k', label='reported mean (misleading)')
ax.legend(); ax.set_title('Summarize a discrete tau by its MODE, not its mean')
plt.tight_layout()